In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import jdex.utils.conventions.paths as pc
import json
from pykeen.triples import TriplesFactory
from rdflib import Graph
import os
from pykeen.pipeline import pipeline
from pykeen.predict import predict_target
import pandas as pd
from tqdm import tqdm
import pandas as pd
import torch
from rdflib import Graph, URIRef

c:\Users\bevia\miniconda3\envs\st_kgs\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
cwd = Path.cwd()
dataset_path = cwd / "WHOW_5_ROFF"
print(dataset_path)

c:\Users\bevia\Documents\GitHub\ST-KGs\WHOW_5_ROFF


In [4]:
with open(dataset_path / pc.INDIVIDUAL_MAPPINGS, "r") as f:
    entity_mapping = json.load(f)

with open(dataset_path / pc.OBJ_PROP_MAPPINGS, "r") as f:
    relation_mapping = json.load(f)

In [ ]:
train_tf = TriplesFactory.from_path(
    dataset_path / pc.TRAIN,
    entity_to_id=entity_mapping,
    relation_to_id=relation_mapping,
)
valid_tf = TriplesFactory.from_path(
    dataset_path / pc.VALID,
    entity_to_id=entity_mapping,
    relation_to_id=relation_mapping,
)

test_tf = TriplesFactory.from_path(
    dataset_path / pc.TEST,
    entity_to_id=entity_mapping,
    relation_to_id=relation_mapping,
)

result = pipeline(
    training=train_tf,
    validation=valid_tf, 
    testing=test_tf,        
    model="TransE",
    model_kwargs=dict(
        embedding_dim=50,
    ),
    optimizer="Adam",
    optimizer_kwargs=dict(
        lr=0.01,
    ),
    training_kwargs=dict(
        num_epochs=3,  
        batch_size=256,
        checkpoint_name='transE-checkpoint.pt',   
        checkpoint_directory=dataset_path /'checkpoints/',    
        checkpoint_on_failure=True
    ),
    
    stopper="early",
    stopper_kwargs=dict(
        frequency=5,       
        patience=2,       
        relative_delta=0.01,
        metric="mrr",     
    ),
    evaluation_kwargs=dict(
        batch_size=256,  
        use_tqdm=True,
    ),
    
    device="gpu",
    random_seed=42,
)

INFO:pykeen.pipeline.api:=> no training loop checkpoint file found at 'c:\Users\bevia\Documents\GitHub\ST-KGs\WHOW_5_ROFF\checkpoints\transE-checkpoint.pt'. Creating a new file.
INFO:pykeen.pipeline.api:Using device: gpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.stoppers.early_stopping:Inferred checkpoint path for best model weights: C:\Users\bevia\.data\pykeen\checkpoints\best-model-weights-23812384-b27c-4c3a-9aff-42a688220de2.pt
INFO:pykeen.training.training_loop:=> no checkpoint found at 'c:\Users\bevia\Documents\GitHub\ST-KGs\WHOW_5_ROFF\checkpoints\transE-checkpoint.pt'. Creating a new file.
Training epochs on cuda:0: 100%|██████████| 3/3 [04:41<00:00, 93.77s/epoch, loss=0.394, prev_loss=0.407]
Evaluating on cuda:0:   6%|▌         | 3.46k/58.5k [01:23<21:22, 42.9triple/s]

State-of-the-Art (SOTA) Knowledge Graph Reasoning (KGR) models that integrate schema axioms are still evaluated using standard ranking metrics (e.g., $MRR$, $Hits@K$). These metrics measure link prediction performance, but they do not prove if a model actually learns and respects ontological rules.The Goal: Use specific Semantic Evaluation Metrics ($Sem@K$, $Inc@K$) to verify if ontology injection genuinely works, comparing a baseline (TransE) with a region-based model (BoxE).

In [12]:
head_predictions = []
tail_predictions = []

with torch.no_grad():
    for h, r, t in tqdm(test_tf.mapped_triples.tolist()[:4], desc="Triple Prediction"):
        df_tail = predict_target(
            model=result.model,
            head=h,
            relation=r,
            triples_factory=train_tf,
        ).df
        top_k_tails = df_tail.sort_values(by="score", ascending=False).head(10)["tail_id"].values
        row_predictions = []
        for i in range(len(top_k_tails)):
            predicted_tail = int(top_k_tails[i])
            row_predictions.append([int(h), int(r), predicted_tail])
        tail_predictions.append(row_predictions)

        df_head = predict_target(
            model=result.model,
            relation=r,
            tail=t,
            triples_factory=train_tf,
        ).df
        top_k_heads = df_head.sort_values(by="score", ascending=False).head(10)["head_id"].values
        row_predictions = []
        for i in range(len(top_k_heads)):
            predicted_head = int(top_k_heads[i])
            row_predictions.append([predicted_head, int(r), int(t)])
        head_predictions.append(row_predictions)

Triple Prediction: 100%|██████████| 4/4 [01:51<00:00, 27.80s/it] 


In [ ]:
from rdflib import RDF, OWL, URIRef
import rdflib
from owlready2 import *
from jdex.owl.reasoning import Reasoner


XSD_NS = "http://www.w3.org/2001/XMLSchema#"
schema_path   = Path(dataset_path) / "tbox" / "schema.owl"
roles_path    = Path(dataset_path) / "tbox" / "taxonomy.owl"
taxonomy_path = Path(dataset_path) / "rbox" / "roles.owl"

def remove_datatype_triples(g):
    to_remove = [
        (s, p, o) for s, p, o in g
        if any(str(x).startswith(XSD_NS) for x in (s, p, o))
        or (p == RDF.type and o == OWL.DatatypeProperty)
    ]
    for triple in to_remove:
        g.remove(triple)
    print(f"  Rimossi {len(to_remove)} triple con datatypes")
    return g

# 1. Carica e pulisci con rdflib
g_schema   = rdflib.Graph().parse(str(schema_path), format="xml")
g_roles    = rdflib.Graph().parse(str(roles_path), format="xml")
g_taxonomy = rdflib.Graph().parse(str(taxonomy_path), format="xml")
g_abox     = rdflib.Graph().parse(str(val_path_nt), format="nt")

print("Pulizia schema:")   ; remove_datatype_triples(g_schema)
print("Pulizia roles:")    ; remove_datatype_triples(g_roles)
print("Pulizia taxonomy:") ; remove_datatype_triples(g_taxonomy)
print("Pulizia abox:")     ; remove_datatype_triples(g_abox)

g_merged = rdflib.Graph()
g_merged += g_schema
g_merged += g_roles
g_merged += g_taxonomy
g_merged += g_abox



merged_path = Path(dataset_path) / "merged_clean.owl"
g_merged.serialize(destination=str(merged_path), format="xml")

win_path = str(merged_path.resolve()).replace("\\", "/")

onto_world = World()

with open(merged_path, "rb") as f:
    merged_onto = onto_world.get_ontology("http://temp.org/merged.owl").load(fileobj=f)

merged_onto.imported_ontologies.clear()

print("Ontologie caricate:")
for onto in onto_world.ontologies.values():
    print(f"  {onto.base_iri}")

reasoner = Reasoner(
            Path().absolute().parent / "reasoners",
            java8_path=Path("C:\Program Files\Java\jdk1.8.0_202"),
            java11_path=Path("C:\Program Files\Java\jdk-11.0.26"),
        )
if not reasoner.consistency(merged_path, verbose=0):
    print("A")


<>:60: SyntaxWarning: invalid escape sequence '\P'
<>:61: SyntaxWarning: invalid escape sequence '\P'
<>:60: SyntaxWarning: invalid escape sequence '\P'
<>:61: SyntaxWarning: invalid escape sequence '\P'
C:\Users\bevia\AppData\Local\Temp\ipykernel_20044\3053193515.py:60: SyntaxWarning: invalid escape sequence '\P'
  java8_path=Path("C:\Program Files\Java\jdk1.8.0_202"),
C:\Users\bevia\AppData\Local\Temp\ipykernel_20044\3053193515.py:61: SyntaxWarning: invalid escape sequence '\P'
  java11_path=Path("C:\Program Files\Java\jdk-11.0.26"),


Pulizia schema:
  Rimossi 3 triple con datatypes
Pulizia roles:
  Rimossi 16 triple con datatypes
Pulizia taxonomy:
  Rimossi 13 triple con datatypes
Pulizia abox:
  Rimossi 0 triple con datatypes
Ontologie caricate:
  http://anonymous/
  http://example.org#
  http://example.org#


In [16]:
from rdflib import RDF, OWL, URIRef
import rdflib
from owlready2 import *
from jdex.owl.reasoning import Reasoner

schema_path = Path(dataset_path) / "ontology.owl" # Core modularized schema
a_box_path = Path(dataset_path) /"abox" / "splits" /"train.nt"
g_schema   = rdflib.Graph().parse(str(schema_path), format="xml")
g_abox     = rdflib.Graph().parse(str(a_box_path), format="nt")

g_merged = rdflib.Graph()
g_merged += g_schema
g_merged += g_abox
merged_path = Path(dataset_path) / "kg_to_verify.owl"
reasoner = Reasoner(
                Path().absolute().parent / "reasoners",
                java8_path=Path("C:\Program Files\Java\jdk1.8.0_202"),
                java11_path=Path("C:\Program Files\Java\jdk-11.0.26"),
            )
g_merged.serialize(destination=str(merged_path), format="xml") 

def verify_consistency(head, relation, tail):
    g_merged.add((head, relation, tail))
    g_merged.serialize(destination=str(merged_path), format="xml") 
    result = reasoner.consistency(merged_path, verbose=0)
    g_merged.remove((head, relation, tail))  
    return result

def verify_consistency_alternative(path, head, relation, tail):
    triple_xml = f'    <rdf:Description rdf:about="{head}">\n        <{relation} rdf:resource="{tail}"/>\n    </rdf:Description>\n'
    with open(path, "r+b") as f:
        content = f.read()
        closing_tag = b"</rdf:RDF>"
        pos = content.rfind(closing_tag)
        if pos == -1:
            raise ValueError("Tag </rdf:RDF> non trovato")
        f.seek(pos)
        f.write(triple_xml.encode("utf-8") + closing_tag)

    result = reasoner.consistency(merged_path, verbose=0)

    with open(path, "r+b") as f:
        content = f.read()
        triple_bytes = triple_xml.encode("utf-8")
    
        if triple_bytes in content:
            head, sep, tail = content.rpartition(triple_bytes)
            content = head + tail       
        f.seek(0)
        f.write(content)
        f.truncate()  
    return result

inc_at_k_head = []
for head_prediction in head_predictions:
    num_inconsistent_triplets = 0
    for triple in head_prediction:
        head_triple = triple[0]
        relation_triple = triple[1]
        tail_triple = triple[2]
        if verify_consistency_alternative(head_triple, relation_triple, tail_triple) == False:
            num_inconsistent_triplets = num_inconsistent_triplets + 1
    inc_at_k_head.append(num_inconsistent_triplets/10)
final_inc_at_k_head = sum(inc_at_k_head) / len(inc_at_k_head)

inc_at_k_tail = []
for tail_prediction in tail_predictions:
    num_inconsistent_triplets = 0
    for triple in tail_prediction:
        head_triple = triple[0]
        relation_triple = triple[1]
        tail_triple = triple[2]
        if verify_consistency_alternative(head_triple, relation_triple, tail_triple) == False:
            num_inconsistent_triplets = num_inconsistent_triplets + 1
    inc_at_k_tail.append(num_inconsistent_triplets/10)
final_inc_at_k_tail = sum(inc_at_k_tail) / len(inc_at_k_tail)
final_inc_at_k = (final_inc_at_k_tail + final_inc_at_k_head) / 2



   




<>:17: SyntaxWarning: invalid escape sequence '\P'
<>:18: SyntaxWarning: invalid escape sequence '\P'
<>:17: SyntaxWarning: invalid escape sequence '\P'
<>:18: SyntaxWarning: invalid escape sequence '\P'
C:\Users\bevia\AppData\Local\Temp\ipykernel_12340\1480487941.py:17: SyntaxWarning: invalid escape sequence '\P'
  java8_path=Path("C:\Program Files\Java\jdk1.8.0_202"),
C:\Users\bevia\AppData\Local\Temp\ipykernel_12340\1480487941.py:18: SyntaxWarning: invalid escape sequence '\P'
  java11_path=Path("C:\Program Files\Java\jdk-11.0.26"),
C:\Users\bevia\AppData\Local\Temp\ipykernel_12340\1480487941.py:17: SyntaxWarning: invalid escape sequence '\P'
  java8_path=Path("C:\Program Files\Java\jdk1.8.0_202"),
C:\Users\bevia\AppData\Local\Temp\ipykernel_12340\1480487941.py:18: SyntaxWarning: invalid escape sequence '\P'
  java11_path=Path("C:\Program Files\Java\jdk-11.0.26"),
C:\Users\bevia\AppData\Local\Temp\ipykernel_12340\1480487941.py:17: SyntaxWarning: invalid escape sequence '\P'
  java8_

TypeError: verify_consistency_alternative() missing 1 required positional argument: 'tail'